[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusicalInformatics/miws26/blob/main/symbolic_tasks_II/rt_fingerprinting/fingerprinting.ipynb)

# Piece Identification

## 1. Introduction

A pianist moves from one piece to another: a few bars of a Mozart sonata, then a Beethoven sonata, and so on. Given the scores of all these pieces, a system has to tell **which piece is being played, as fast as possible**. This is **piece identification** (and, as a by-product, finding the **position in the score**). A score following system needs it before it can start to follow the score.

We work with **symbolic data**: the system does not hear audio, it receives the notes that the pianist plays (like the output of a MIDI keyboard).

* **Input:** a stream of notes (pitch and onset time) that switches from one piece to another, and the scores of the possible pieces.
* **Output:** the piece that is being played, after every **query**, i.e., a window of the last notes. The system works **online**: it can only use the notes it has already heard.

What makes it hard:

* The pianist plays **expressively** (tempo changes, early or late notes, wrong and extra notes). The score has none of that.
* The system does not know when the pianist switches piece, or where in the new piece they start.
* It has to answer after a few notes, and be **fast**.
* In the challenge, the pianist can also change the **key and the mode** (e.g., a piece in major is played in minor).

We introduce a classic solution, **symbolic fingerprinting** (Arzt, Böck and Widmer, 2012). Like audio fingerprinting systems that identify songs, it converts the music into **tokens** and looks up the tokens of what is being played in a **hash table** built from the tokens of all the scores.

In [ ]:
import os
import sys

try:
    import google.colab

    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    ! pip install partitura

    if not os.path.exists("./miws26"):
        !git clone https://github.com/MusicalInformatics/miws26.git
    NOTEBOOK_DIR = os.path.abspath("./miws26/symbolic_tasks_II/rt_fingerprinting")
else:
    NOTEBOOK_DIR = os.path.abspath(".")

sys.path.insert(0, NOTEBOOK_DIR)

In [ ]:
import warnings

import matplotlib.pyplot as plt
from IPython.display import HTML

from challenge import evaluation, format_evaluation
from fingerprinting import (
    OnlineFingerprinting,
    build_hash_table_v2,
    generate_tokens,
    match_histograms,
    rank_matches,
)
from fingerprinting_plots import (
    animate_tokenization,
    playback,
    plot_matches,
    simple_score,
)
from utils import (
    dataset_scores,
    load_note_array,
    load_scores,
    load_sequences,
    piece_name,
    unpack_data,
)

warnings.filterwarnings("ignore")

## 2. Data

The data is not in the repository: **download it from Moodle** (a zip file with the folders `scores`, `train` and `test`).

* `scores`: the MusicXML scores of the **85 pieces** that the pianist can play (Mozart and Beethoven sonatas, and a few pieces of the Vienna 4x22 corpus), and `scores.csv` with the name of each piece and whether its repeats have to be written out (*unfolded*), because the pianist plays them.
* `train`: 150 **sequences** as MIDI files, **with ground truth**. Each sequence simulates a pianist who plays 5 to 10 segments of different pieces, one after the other. A segment is a slice of 5 to 30 seconds of a real performance. A table (`..._segments.csv`) gives the start, the end and the piece of each segment, and another one (`..._notes.csv`) gives, for each note, the score note it is aligned to. Some segments are **transposed** or change **mode** (major to minor, or the other way around).
* `test`: 50 sequences without ground truth, for the challenge server.

In the next cell, replace the path with the location of the zip file. The function `unpack_data` unpacks it in the folder `data`. If the folder already exists, it only unpacks again when some files are missing (`metadata.csv` lists them).

In [ ]:
ZIP_PATH = "path/to/fingerprinting_dataset.zip"  # replace this with the path of the file that you downloaded from Moodle

DATA_DIR = unpack_data(ZIP_PATH, os.path.join(NOTEBOOK_DIR, "data"))

In [ ]:
score_files, unfold = dataset_scores(DATA_DIR)
sequences = load_sequences(os.path.join(DATA_DIR, "train"))
print(len(score_files), "scores,", len(sequences), "sequences in the training set")

notes, segments, note_map = sequences[1]
print(len(notes), "notes")
print(segments[["start", "end", "piecename", "transposition", "mode_change"]])

### 2.1 Piano roll and audio of a sequence

The notes of a sequence are a [Partitura](https://github.com/CPJKU/partitura) note array (`utils.load_note_array`). The player below shows the **piano roll** ([`partitura.utils.compute_pianoroll`](https://partitura.readthedocs.io/en/latest/modules/partitura.utils.html#partitura.utils.compute_pianoroll)) of the first minute of the sequence and plays its **audio**. Press play: the blue **playhead** moves over the piano roll, and when it crosses a red line the pianist switches to another piece. The name of the piece that is playing is shown below the player. You can also click on the piano roll to jump to that time.

The audio is synthesized with Partitura. With [fluidsynth](https://www.fluidsynth.org/) it uses a sampled piano sound (`partitura.utils.fluidsynth.synthesize_fluidsynth`); otherwise it uses additive synthesis (`partitura.utils.synth.synthesize`), which sounds more artificial. The function `synthesize_notes` in `fingerprinting_plots.py` picks the one that is available. (The player needs JavaScript, so use Jupyter or Colab.)

In [ ]:
playback(notes, segments, start=0, end=60)

## 3. Symbolic fingerprinting

### 3.1 Tokens

A fingerprint describes a piece with **small patterns** that can be found again in a performance. They have to be **robust**: the same in the score and in a performance, even at another tempo. A **token** is made of three notes:

* the three **pitches** (pitch 1, pitch 2, pitch 3), and
* the **ratio** of the time differences between the notes, $(	ext{onset}_2 - 	ext{onset}_1) / (	ext{onset}_3 - 	ext{onset}_2)$, rounded to an integer. It does not change if the passage is played faster or slower.

Tokens are generated for each note of the piece (the *first note*):

1. The *second note* is any of the next $n_1$ notes after the first note.
2. The *third note* is any of the next $n_2$ notes after the second note.
3. The notes of a token have to be at least $d$ = 0.05 seconds apart (so that the notes of a chord are not used together).

With $n_1 = n_2 = 5$, each note is the first note of up to 25 tokens. A piece has many tokens, so even if some notes are wrong or missing in a performance, many tokens are still correct.

### 3.2 Tokens of the scores and of the performances

Scores and performances use the same function (`generate_tokens`). Only the notes differ:

* **Score:** all the notes, with onsets in seconds computed with a constant tempo (120 bpm). If the pianist plays the repeats, the score is unfolded (`unfold_part_maximal` of Partitura). Each token is stored in the hash table with the piece, its *position* in the score (the onset of its first note) and the time difference between its first two notes.
* **Performance (query):** the last notes that have been played, with their onsets in seconds. The tempo and the expressive timing differ from the score, so the ratios will not be exactly the same. A token of the score therefore matches a token of the query if the pitches are equal and the ratios differ by at most 25%.

In [ ]:
score_notes = load_note_array(
    os.path.join(DATA_DIR, "scores", "Beethoven_Sonata_8-3.musicxml"), unfold=True
)
print("Score:")
print(score_notes[["onset_sec", "duration_sec", "pitch"]][:5])
print(generate_tokens(score_notes, n1=5, n2=5)[:3], "...")

print("\nPerformance (first segment of the sequence):")
print(notes[["onset_sec", "duration_sec", "pitch"]][:5])
print(generate_tokens(notes[:30], n1=5, n2=5)[:3], "...")

Each token is `((pitch 1, pitch 2, pitch 3, ratio), onset of the first note)`.

### 3.3 Animation of the tokens

The animation shows how tokens are generated on a **very simple example** (a melody of 7 notes, with $n_1 = n_2 = 2$). In each step, the first note (red), a second note (blue) and a third note (green) are selected. The arrows are the time differences between the notes, and the list below the piano roll has the tokens generated so far. Use the controls to go through the steps.

In [ ]:
anim = animate_tokenization(simple_score(), n1=2, n2=2, interval=1200)
HTML(anim.to_jshtml())

### 3.4 Hash table

The hash table has the tokens of **all the scores**. It maps the pitches and the (rounded) ratio of a token to the pieces and positions where the token appears. Building it for the 85 scores takes about a minute.

In [ ]:
hash_table = build_hash_table_v2(score_files, n1=5, n2=5, unfold=unfold)
print(len(hash_table), "tokens with position")

### 3.5 Identifying the piece and the position (Figure 3 of the paper)

To identify a query we generate its tokens and look up the matching tokens in the hash table. Plot each match as its **score time** (the position in the piece) against its **query time**. If the query comes from a place in the piece, its matches lie on a **diagonal**: they all have the same offset between score time and query time. The matches of other pieces are random coincidences, so they are scattered (panel a, zoomed on the part of the score where the peak is). The matches of the correct piece in the highest bin are red: that is the diagonal.

Finding a diagonal directly would be expensive. Instead, each match goes into a **histogram** (one per piece, with bins of 1 second) according to the **difference between score time and query time**. A diagonal falls into a single bin, so it becomes a **peak** (panel b). The **tempo** of the pianist can differ from the tempo of the score, so the query time of a token is first multiplied by the tempo ratio of the two tokens (the time difference between the first two notes of the score token divided by that of the query token).

The score of a piece is the number of tokens in its highest bin. The piece with the highest score is the identified piece, and its bin gives the position in the score. The figure uses the first 60 notes of the sequence above (the first piece is Beethoven's Piano Sonata No. 8, third movement).

In [ ]:
query = notes[:60]
fig = plot_matches(query, hash_table, piece=segments["piecename"][0])
plt.show()

for piece, position, n_tokens in rank_matches(match_histograms(query, hash_table))[:3]:
    print(f"{piece:25s} score time {position:6.1f} s, {n_tokens} tokens in the bin")

### 3.6 Online identification

In the challenge the system receives the notes **one by one**. `OnlineFingerprinting` keeps a sliding window with the last notes (`window_size`). When a note arrives, it creates only the tokens that end with that note, looks them up in the hash table, and keeps the matches while the first note of their token is still in the window. After every `window_size` notes (the query) it computes the histograms and returns the ranking of pieces and positions (`rankings`, `matches`). This is the system we evaluate next.

## 4. Evaluation

We evaluate with the metrics of **Table 3** of the paper, on the training sequences (they have ground truth). Each sequence is split into queries of 5, 10, 20, 30, 40, 50 and 60 notes. For each query length the table reports:

* the percentage of queries where the **correct piece** is the top match, or among the top 2 to 5. The correct piece of a query is the one playing at its last note,
* the same for the **position**: it is correct if it is within 1.5 seconds of the real position, or at a place of the score with identical content (a repeated passage is not an error),
* the mean **duration** of the queries (in seconds) and their **number**. (The paper also reports the execution time, but it includes the transcription of audio, and we work with symbolic data.)

The first table uses **all the queries**, including those where the pianist switches pieces (the notes come from two pieces). The second one only uses the queries inside a segment. This system **cannot handle a change of key or mode**, so the transposed segments are expected to fail: that is the difficulty of the challenge. The evaluation uses `N_SEQUENCES` sequences and takes a few minutes (you can use all 150).

In [ ]:
N_SEQUENCES = 50
scores = load_scores(score_files, unfold)

results = evaluation(
    sequences[:N_SEQUENCES], hash_table, model=OnlineFingerprinting, scores=scores
)
print("All the queries")
print(format_evaluation(results))

In [ ]:
results = evaluation(
    sequences[:N_SEQUENCES],
    hash_table,
    model=OnlineFingerprinting,
    scores=scores,
    only_single_piece=True,
)
print("Only the queries inside a segment")
print(format_evaluation(results))

## References

* Andreas Arzt, Sebastian Böck and Gerhard Widmer (2012). [Fast Identification of Piece and Score Position via Symbolic Fingerprinting](https://archives.ismir.net/ismir2012/paper/000433.pdf). In *Proceedings of the 13th International Society for Music Information Retrieval Conference (ISMIR)*, Porto, Portugal.